In [80]:
# 1) Imports & data
import time
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

# load your train/val/test sets
X_train, y_train = joblib.load("data/processed/train.pkl")
X_val,   y_val   = joblib.load("data/processed/val.pkl")
X_test,  y_test  = joblib.load("data/processed/test.pkl")

# merge train+val for final training
import numpy as np
X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

# 2) Fit the booster
model = HistGradientBoostingRegressor(
    max_iter=200,        # number of trees
    learning_rate=0.1,   # shrinkage
    max_depth=10,        # typical good default
    random_state=42,
    verbose=1
)
t0 = time.time()
model.fit(X_train_full, y_train_full)
fit_time = time.time() - t0

# 3) Evaluate on test
t0 = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - t0

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2   = r2_score(y_test, y_pred)

# 4) Print results
print(f"🕑  Training time: {fit_time:.2f} s")
print(f"🔍  Test RMSE     : {test_rmse:,.2f}")
print(f"📈  Test R²       : {test_r2:.3f}")
print(f"⏱️  Inference time: {pred_time:.3f} s on {len(X_test)} samples")

Binning 0.190 GB of training data: 0.682 s
Binning 0.021 GB of validation data: 0.009 s
Fitting gradient boosted rounds:
[1/200] 1 tree, 31 leaves, max depth = 9, train loss: 311951035.49775, val loss: 252305946.29686, in 0.050s
[2/200] 1 tree, 31 leaves, max depth = 10, train loss: 272049898.13698, val loss: 216385934.99918, in 0.037s
[3/200] 1 tree, 31 leaves, max depth = 10, train loss: 239362328.66072, val loss: 186880641.04861, in 0.037s
[4/200] 1 tree, 31 leaves, max depth = 8, train loss: 212732331.60340, val loss: 163875633.29728, in 0.032s
[5/200] 1 tree, 31 leaves, max depth = 10, train loss: 190490900.88634, val loss: 143559242.41667, in 0.037s
[6/200] 1 tree, 31 leaves, max depth = 9, train loss: 172144511.64466, val loss: 127209750.83630, in 0.034s
[7/200] 1 tree, 31 leaves, max depth = 10, train loss: 156991210.22780, val loss: 114665872.36246, in 0.036s
[8/200] 1 tree, 31 leaves, max depth = 10, train loss: 144077495.10849, val loss: 102931035.90790, in 0.035s
[9/200] 1 

In [84]:
!pip install lightgbm --quiet

In [90]:
import time
import numpy as np
import joblib
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.metrics import mean_squared_error, r2_score

# 1) Load your pre‐split, preprocessed data
X_train, y_train = joblib.load("data/processed/train.pkl")
X_val,   y_val   = joblib.load("data/processed/val.pkl")
X_test,  y_test  = joblib.load("data/processed/test.pkl")

# 2) Merge train + val for final fit
X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

# 3) Instantiate the model
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
)

# 4) Fit with early stopping via callbacks
t0 = time.time()
model.fit(
    X_train_full, y_train_full,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[
        early_stopping(stopping_rounds=50),  # stop if no improvement in 50 rounds
        log_evaluation(period=100),         # print eval results every 100 iters
    ],
)
fit_time = time.time() - t0

# 5) Evaluate on test set, timing the predict call
t1 = time.time()
y_pred = model.predict(X_test)
pred_time = time.time() - t1

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2   = r2_score(y_test, y_pred)

# 6) Report everything
print(f"⏱️  Training time   : {fit_time:.2f} s")
print(f"🔍  Test  RMSE      : {test_rmse:,.2f}")
print(f"📈  Test  R²        : {test_r2:.3f}")
print(f"⏲️  Predict on {len(X_test)} samples: {pred_time:.3f} s")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013864 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1665
[LightGBM] [Info] Number of data points in the train set: 172658, number of used features: 130
[LightGBM] [Info] Start training from score 7000.110067
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 9130.87	valid_0's l2: 8.33728e+07
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[200]	valid_0's rmse: 8036.37	valid_0's l2: 6.45833e+07
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

In [92]:
!pip install xgboost --quiet

In [111]:
import time
import warnings
import numpy as np
import joblib
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# 1) load your preprocessed splits
X_train, y_train = joblib.load("data/processed/train.pkl")
X_val,   y_val   = joblib.load("data/processed/val.pkl")
X_test,  y_test  = joblib.load("data/processed/test.pkl")

# 2) merge train+val for final fit
X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

# 3) create DMatrix objects
dtrain = xgb.DMatrix(X_train_full, label=y_train_full)
dval   = xgb.DMatrix(X_val,       label=y_val)
dtest  = xgb.DMatrix(X_test,      label=y_test)

# 4) specify your params (including integer 'verbosity')
params = {
    "tree_method":    "hist",
    "eta":            0.05,
    "max_depth":      8,
    "objective":      "reg:squarederror",
    "eval_metric":    "rmse",
    "verbosity":      0,      # <--- mute all messages except eval logs
    "seed":           42,
}

# 5) (optional) still filter Python warnings
warnings.filterwarnings("ignore")

# 6) train with early stopping
t0 = time.time()
bst = xgb.train(
    params,
    dtrain,
    num_boost_round       = 1000,
    evals                  = [(dval, "valid")],
    early_stopping_rounds = 50,
    verbose_eval          = 50,   # prints one line every 50 rounds
)
train_time = time.time() - t0

# 7) inference timing
t1     = time.time()
y_pred = bst.predict(dtest)
pred_time = time.time() - t1

# 8) final metrics
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
test_r2   = r2_score(y_test, y_pred)

# 9) clean summary
print(f"⏱  Training     : {train_time:.2f}s")
print(f"🔎  Test RMSE    : {test_rmse:,.2f}")
print(f"📈  Test R²      : {test_r2:.3f}")
print(f"⏱  Predict time : {pred_time:.3f}s for {len(X_test)} samples")

[0]	valid-rmse:24759.31765
[50]	valid-rmse:6871.40664
[100]	valid-rmse:4368.16010
[150]	valid-rmse:3475.56462
[200]	valid-rmse:2931.01152
[250]	valid-rmse:2621.55414
[300]	valid-rmse:2375.10640
[350]	valid-rmse:2205.56504
[400]	valid-rmse:2024.98744
[450]	valid-rmse:1918.17548
[500]	valid-rmse:1769.28266
[550]	valid-rmse:1620.76377
[600]	valid-rmse:1505.03804
[650]	valid-rmse:1394.64705
[700]	valid-rmse:1314.04597
[750]	valid-rmse:1222.83434
[800]	valid-rmse:1156.28365
[850]	valid-rmse:1093.01546
[900]	valid-rmse:1028.66677
[950]	valid-rmse:970.18195
[999]	valid-rmse:920.46333
⏱  Training     : 8.87s
🔎  Test RMSE    : 6,270.54
📈  Test R²      : 0.930
⏱  Predict time : 0.148s for 43165 samples
